# Transfer Learning / Fine-tuning Example (Tabular)

## From regression to classification on the same domain (Housing)

### Idea
1) Train a **base regression model** to predict house value.
2) Reuse the learned **feature representation** for a **new job**: predict whether a house is **"expensive"** (top quartile).
3) Do it in two stages:
   - Train a new classification head with the base frozen.
   - Unfreeze the top layers and fine-tune with a lower learning rate.

This is a practical form of transfer learning for tabular data.

---


In [ ]:
# %% [code]
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, roc_auc_score, accuracy_score

import tensorflow as tf

RANDOM_STATE: int = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)


## 1) Load + preprocess data (same features for both tasks)

In [ ]:
# %% [code]
def load_california_housing_df() -> pd.DataFrame:
    """Load California housing dataset and rename target to `price` for clarity."""
    data = fetch_california_housing(as_frame=True)
    df = data.frame.copy()
    df.rename(columns={"MedHouseVal": "price"}, inplace=True)
    return df

def add_basic_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add a few stable ratio features to improve representation learning."""
    out = df.copy()
    eps = 1e-6
    out["RoomsPerOccup"] = out["AveRooms"] / (out["AveOccup"] + eps)
    out["BedrmsPerRoom"] = out["AveBedrms"] / (out["AveRooms"] + eps)
    out["PopPerOccup"] = out["Population"] / (out["AveOccup"] + eps)
    return out

def make_numeric_preprocessor(num_cols: list[str]) -> ColumnTransformer:
    """Create a numeric-only preprocessing pipeline (impute + scale)."""
    return ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), num_cols)
    ])

def to_dense(x: Any) -> np.ndarray:
    """Convert sparse matrix to dense numpy array if needed."""
    return x.toarray() if hasattr(x, "toarray") else np.asarray(x)

df = add_basic_features(load_california_housing_df())
X = df.drop(columns=["price"])
y_price = df["price"].astype(np.float32).values

X_train, X_valid, y_price_train, y_price_valid = train_test_split(
    X, y_price, test_size=0.2, random_state=RANDOM_STATE
)

num_cols = X.columns.tolist()
pre = make_numeric_preprocessor(num_cols)

Xtr = to_dense(pre.fit_transform(X_train))
Xva = to_dense(pre.transform(X_valid))

Xtr.shape, Xva.shape


## 2) Train base model (Regression)

In [ ]:
# %% [code]
def build_base_regressor(input_dim: int) -> tf.keras.Model:
    """Build a base regression MLP with a named feature layer for transfer.

    The layer named `feature_layer` is used as the reusable representation.

    Args:
        input_dim: Number of input features.

    Returns:
        Compiled Keras regression model.
    """
    inputs = tf.keras.Input(shape=(input_dim,))
    x = tf.keras.layers.Dense(128, activation="relu")(inputs)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dense(32, activation="relu", name="feature_layer")(x)
    outputs = tf.keras.layers.Dense(1, name="price_output")(x)

    model = tf.keras.Model(inputs, outputs, name="base_price_regressor")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="mse",
        metrics=[tf.keras.metrics.RootMeanSquaredError(name="rmse")],
    )
    return model

base_model = build_base_regressor(Xtr.shape[1])
base_model.summary()

history = base_model.fit(
    Xtr, y_price_train,
    validation_data=(Xva, y_price_valid),
    epochs=40,
    batch_size=128,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor="val_rmse", mode="min", patience=8, restore_best_weights=True)
    ],
    verbose=0,
)

pred_price = base_model.predict(Xva, verbose=0).ravel()
rmse = mean_squared_error(y_price_valid, pred_price, squared=False)
print("Base regression VALID RMSE:", rmse)


## 3) New prediction job: classification (Expensive vs Not)

We define **expensive** as the top 25% of training prices, then:
- Freeze the representation layers.
- Train a new classification head.
- Unfreeze the top representation layer(s) and fine-tune with a smaller learning rate.

In [ ]:
# %% [code]
def make_expensive_labels(y_train: np.ndarray, y_valid: np.ndarray, percentile: float = 75.0) -> Tuple[np.ndarray, np.ndarray, float]:
    """Create binary labels based on whether a value is above a percentile threshold.

    Args:
        y_train: Training target values.
        y_valid: Validation target values.
        percentile: Percentile threshold (e.g., 75.0 => top quartile).

    Returns:
        (y_train_bin, y_valid_bin, threshold)
    """
    thr = float(np.percentile(y_train, percentile))
    return (y_train >= thr).astype(int), (y_valid >= thr).astype(int), thr

y_cls_train, y_cls_valid, thr = make_expensive_labels(y_price_train, y_price_valid, percentile=75.0)
print("Expensive threshold:", thr)
print("Train positive rate:", y_cls_train.mean(), "| Valid positive rate:", y_cls_valid.mean())


In [ ]:
# %% [code]
def build_transfer_classifier(
    base_regressor: tf.keras.Model,
    input_dim: int,
    freeze_base: bool = True,
    head_units: int = 16,
    lr: float = 1e-3,
) -> tf.keras.Model:
    """Build a transfer-learning classifier using a pretrained regressor's feature layer.

    Args:
        base_regressor: Pretrained regression model containing `feature_layer`.
        input_dim: Number of input features.
        freeze_base: Whether to freeze the base feature extractor.
        head_units: Units in the new classification head.
        lr: Learning rate.

    Returns:
        Compiled Keras classification model.
    """
    feature_extractor = tf.keras.Model(
        inputs=base_regressor.input,
        outputs=base_regressor.get_layer("feature_layer").output,
        name="feature_extractor",
    )

    feature_extractor.trainable = not freeze_base

    inputs = tf.keras.Input(shape=(input_dim,))
    x = feature_extractor(inputs)
    x = tf.keras.layers.Dense(head_units, activation="relu")(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="expensive_output")(x)

    clf = tf.keras.Model(inputs, outputs, name="expensive_classifier")
    clf.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
    )
    return clf

# Stage 1: train new head (base frozen)
clf_frozen = build_transfer_classifier(base_model, input_dim=Xtr.shape[1], freeze_base=True, head_units=16, lr=1e-3)

hist1 = clf_frozen.fit(
    Xtr, y_cls_train,
    validation_data=(Xva, y_cls_valid),
    epochs=30,
    batch_size=128,
    verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=6, restore_best_weights=True)],
)

proba1 = clf_frozen.predict(Xva, verbose=0).ravel()
pred1 = (proba1 >= 0.5).astype(int)
print("Stage 1 VALID AUC:", roc_auc_score(y_cls_valid, proba1))
print("Stage 1 VALID ACC:", accuracy_score(y_cls_valid, pred1))


## 4) Fine-tune: unfreeze top layers and train with smaller LR

In [ ]:
# %% [code]
def unfreeze_top_layers(model: tf.keras.Model, n_layers_to_unfreeze: int = 2) -> None:
    """Unfreeze the last N layers of a model (in-place).

    Args:
        model: A Keras model.
        n_layers_to_unfreeze: Number of last layers to set trainable=True.
    """
    if n_layers_to_unfreeze <= 0:
        return
    for layer in model.layers[-n_layers_to_unfreeze:]:
        layer.trainable = True

# Copy the frozen model structure (we can just reuse clf_frozen and unfreeze extractor inside it)
# Unfreeze the feature extractor's last layers by locating it:
feature_extractor_layer = clf_frozen.get_layer("feature_extractor")

# Unfreeze last 2 layers of the feature extractor (Dense layers near the top)
unfreeze_top_layers(feature_extractor_layer, n_layers_to_unfreeze=2)

# Re-compile with smaller LR (important for fine-tuning)
clf_frozen.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
)

hist2 = clf_frozen.fit(
    Xtr, y_cls_train,
    validation_data=(Xva, y_cls_valid),
    epochs=25,
    batch_size=128,
    verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=6, restore_best_weights=True)],
)

proba2 = clf_frozen.predict(Xva, verbose=0).ravel()
pred2 = (proba2 >= 0.5).astype(int)
print("Stage 2 (fine-tuned) VALID AUC:", roc_auc_score(y_cls_valid, proba2))
print("Stage 2 (fine-tuned) VALID ACC:", accuracy_score(y_cls_valid, pred2))


## Notes
- This example is **self-contained**: it trains a base model and then transfer-learns.
- In production you would usually **save** the base model and load it in a separate pipeline.
- You can extend this with Optuna to tune: which layers to unfreeze, LR schedule, head size, etc.


## Summary: Scratch vs Transfer Learning
This cell compares **training from scratch**, **frozen transfer**, and **fine‑tuned transfer**.

In [ ]:

summary = {
    "scratch": {
        "auc": roc_auc_score(y_cls_valid, scratch_proba),
        "accuracy": accuracy_score(y_cls_valid, scratch_pred),
    },
    "transfer_frozen": {
        "auc": roc_auc_score(y_cls_valid, proba1),
        "accuracy": accuracy_score(y_cls_valid, pred1),
    },
    "transfer_finetuned": {
        "auc": roc_auc_score(y_cls_valid, proba2),
        "accuracy": accuracy_score(y_cls_valid, pred2),
    },
}
summary
